Mandatory assignement 

1a: Recover the private key and plaintext

Brute force approach:

In [20]:

# public parameters:
p = 29837
g = 42
pk = 22690 

# c1 and c2 has been itercepted:
c1 = 23447
c2 = 8372

def recover(p, c1, c2):

    # find exponent x - the private key 
    for i in range(p-1):
        if (pow(g, i, p) == pk):
            x = i
            break
    
    # find m
    c1_ = pow(c1,-x,p) # -x is inverse
    m = (c2*c1_) % p # Modular division means multiplying by an inverse
    
    return m

print(recover(p, c1, c2))

26000


**1b. Modify the ciphertext without the private key**

The fictional target identifier is m' = 12345.

Goal is to modify c2 that the message become: 12345


**Step 1**: 
We know from the nature of ElGamal that c2 = m · PK^r mod p.

**Step 2**: 
I want to define a new c2' still using PK^r, but with, m' in stead of m. That is:
c2' ≡ m' * PK^r (mod p)

**Step 3**: 
Here I find a multiplier k

Say c2' = k * c2. 

Then
k * m * PK^r ≡ m' * PK^r (mod p)

PK^r cancels from both sides:

k * m ≡ m' (mod p)

k ≡ m' * m (mod p) #Modular division means multiplying by an inverse

Now, k only depends on m, m' and p, which are all allowed inputs.


In [ ]:
m = 26000
m_target = 12345 # target identifier

# change the cyphertext to 12345
def attack(p, c1, c2, m, m_target):
    m_inverse = pow(m, -1, p)   # modular inverse of m,
    k = (m_target * m_inverse) % p  # multply by the target
    c2_target = (k * c2) % p
    return c1, c2_target

c1_, c2_ = attack(p, c1, c2, m, m_target)

# Verification using the private key x from 1a:
print(recover(p, c1_, c2_))



12345


1b: 
* Give modified cipertext
* justify your construction algebraically
* may use the recovered private key afterwards to verify the result

reflection: 
(elgamal beskytter ikke man in the middle attack)


**Exercise 2: Hash me if you can**

Context:
* UTF-8 is an "encryption" for turning characters into bytes 
* Unicode Transformation Format 8-bit
* text -> bytes

In [ ]:
from hashlib import sha256

# getting to know the 3 versions of digest and sha256

# adding bits changes the length of the truncated digest 
digest = sha256("hello".encode("utf-8")).digest()  # 32 bytes
h16 = digest[:2]   # first 2 bytes / 16 bits
h24 = digest[:3]   # first 3 bytes / 24 bits
h32 = digest[:4]   # first 4 bytes / 32 bits

print(h16.hex())  # 2cf2
print(h24.hex())  # 2cf24d
print(h32.hex())  # 2cf24dba

# Changing a single character scrambles the output completely
input1 = sha256("johan".encode("utf-8")).digest()  # 32 bytes
input2 = sha256("johann".encode("utf-8")).digest()  # 32 bytes

print((input1[:4]).hex()) # d9a31550
print((input2[:4]).hex()) # a27d6fe2





2cf2
2cf24d
2cf24dba
d9a31550
a27d6fe2


**2a** Implement a collision search:

For each output length, find distinct messages with the same truncated digest

In [ ]:
def find_collision(num_bytes):
    messages = {} #dictionary of truncated hash and messages
    i = 0

    while True:
        i += 1
        newMsg = "msg" + str(i)
        h = sha256(newMsg.encode("utf-8")).digest()[:num_bytes]
        if h in messages:
            return i, newMsg, messages[h]
        else:
            messages[h] = newMsg

# Test with the three levels of digest bits:
for n in (2, 3, 4):
    searches, m1, m2 = find_collision(n)
    print (
        n, "bits:", 
        m1, m2,
        "searches:", searches)

2 bits: msg121 msg114 searches: 121
4 bits: msg151340 msg39097 searches: 151340
6 bits: msg10296150 msg2799261 searches: 10296150


**2b** Distinguish collision and second-preimage resistance. 

* How does the task change if m1 is fixed in advance and you must find a different m2 with the same hash? 

Instead of finding any two messages that collide, you must match one specific, already-chosen hash value H_b(m₁). The birthday effect disappears: you can't compare new messages against each other, only against that single target. Each new message m₂ is an independent trial that succeeds with probability 1/2^b, so you hash candidates one at a time and check whether each equals the target, with no benefit from storing earlier results.

| Setting | Expected work | b = 16 | b = 24 | b = 32 |
|---|---|---|---|---|
| Collision (any pair) | ≈ 2^(b/2) | ~2⁸ | ~2¹² | ~2¹⁶ |
| fixed m₁ | ≈ 2^b | 2¹⁶ | 2²⁴ | 2³² |

* Compare the expected work for generic attacks in the two settings. 
(You do not need to implement a second-preimage search)

* Does your experiment show that full SHA-256 is practically broken?

The second-preimage attack is 2^(b/2) times more expensive. For b = 32, that is roughly 65 thousand hashes versus about 4.3 billion. The collision attack also needs memory to store about 2^(b/2) hashes, while the second-preimage search needs essentially none.

For full SHA-256 (b = 256) the gap is 2¹²⁸ for collisions versus 2²⁵⁶ for second preimages, both far out of reach, which supports your answer that the experiment does not show SHA-256 is broken.


4:
use filter and time, to construct what is happening

**Exercise 3: Breaking BAD** 

Tip: 2nd exercise, consider the rotation property



